# A Safe First Pokémon Agent: Legal Moves, Deck Math, One Archive

This starter makes the simulation interface visible:

- a 60-card deck;
- a legal option ranking for ordinary selection states;
- explicit handling of the turn-order choice (`context == 41`);
- a packager for `main.py`, `agent.py`, and `deck.csv`.

It is designed as a transparent control policy for later behavior cloning or
PPO, not as a claim of a top leaderboard score.

## Reproducible starter

In [1]:
"""Runnable legal-action starter for the Pokémon TCG Playground.

The public starter interface is:
  - observation["select"] is None: return the 60-card deck;
  - otherwise choose between minCount and maxCount options.

This heuristic is deliberately simple.  It is a safe base for behavior cloning
or PPO, and its decisions are easy to inspect.
"""

from pathlib import Path
import csv
import math
import random
import tarfile

DECK_IDS = [
    5, 5, 13, 19, 19, 19, 19, 66, 66, 66, 140,
    305, 305, 305, 305, 741, 741, 741, 741,
    742, 742, 742, 742, 743, 743, 743,
    1079, 1079, 1079, 1079, 1081, 1081,
    1086, 1086, 1086, 1086, 1097, 1097, 1097,
    1129, 1152, 1152, 1152, 1152,
    1182, 1182, 1182, 1184, 1197, 1197,
    1225, 1225, 1225, 1225, 1231, 1231, 1231, 1231,
    1264, 1264,
]

# Type ranks are intentionally coarse.  Replace them with an action-value model.
PRIORITY = {
    8: 1000,   # basic energy
    9: 850,    # evolution / setup
    7: 700,    # play Pokemon
    10: 550,   # useful item/tool
    13: 300,   # draw/search
    14: 1,     # low-value forced move
    12: -5,
}


def agent(obs_dict):
    select = obs_dict.get("select")
    if select is None:
        return list(DECK_IDS)
    options = select.get("option") or []
    if not options:
        return []
    max_count = int(select.get("maxCount", 1) or 0)
    min_count = int(select.get("minCount", max_count) or 0)
    if max_count == 0:
        return []

    # Context 41 is the player-chosen turn order.
    if int(select.get("context", 0) or 0) == 41:
        for i, option in enumerate(options):
            if int(option.get("type", -1)) == 1:
                return [i]

    scored = [
        (PRIORITY.get(int(o.get("type", -1)), 50), -i, i)
        for i, o in enumerate(options)
    ]
    scored.sort(reverse=True)
    count = max(min_count if min_count > 0 else 1, min(max_count, len(options)))
    return [item[2] for item in scored[:count]]


def deck_quality(deck):
    basics = len(deck)
    p_mulligan = math.comb(60 - basics, 7) / math.comb(60, 7)
    return p_mulligan if basics else 1.0


AGENT_SOURCE = '''from pathlib import Path
import math

DECK_IDS = {deck_ids!r}
PRIORITY = {priority!r}


def agent(obs_dict):
    select = obs_dict.get("select")
    if select is None:
        return list(DECK_IDS)
    options = select.get("option") or []
    if not options:
        return []
    max_count = int(select.get("maxCount", 1) or 0)
    min_count = int(select.get("minCount", max_count) or 0)
    if max_count == 0:
        return []
    if int(select.get("context", 0) or 0) == 41:
        for i, option in enumerate(options):
            if int(option.get("type", -1)) == 1:
                return [i]
    scored = [
        (PRIORITY.get(int(o.get("type", -1)), 50), -i, i)
        for i, o in enumerate(options)
    ]
    scored.sort(reverse=True)
    count = max(min_count if min_count > 0 else 1, min(max_count, len(options)))
    return [item[2] for item in scored[:count]]
'''.format(deck_ids=DECK_IDS, priority=PRIORITY)


if __name__ == "__main__":
    root = Path("submission_agent")
    root.mkdir(exist_ok=True)
    (root / "main.py").write_text(
        "from agent import agent\n", encoding="utf-8"
    )
    (root / "agent.py").write_text(AGENT_SOURCE, encoding="utf-8")
    with (root / "deck.csv").open("w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["card_id"])
        writer.writerows(([card] for card in DECK_IDS))
    with tarfile.open("submission.tar.gz", "w:gz") as archive:
        archive.add(root / "main.py", arcname="main.py")
        archive.add(root / "agent.py", arcname="agent.py")
        archive.add(root / "deck.csv", arcname="deck.csv")
    print("wrote submission.tar.gz")

    examples = [
        {"select": {"context": 41, "option": [{"type": 0}, {"type": 1}]}},
        {"select": {"maxCount": 2, "minCount": 1, "option": [{"type": 8}, {"type": 12}]}},
    ]
    print([agent(x) for x in examples])

# Before leaderboard submission, preserve the official cg/ and model files by
# starting from the competition starter archive and replacing only agent files.


wrote submission.tar.gz
[[1], [0, 1]]


## Make it competitive

Start from a public runnable archive, preserve its `cg/` and model files, and
replace only the agent logic.  Evaluate against several distinct public bots
with Best-of-3 matches, then retain a candidate only when its confidence
interval supports the change.

## Credits

Data, rules and submission format belong to the competition host. Cite every public notebook, dataset or pretrained model used in later versions.